In [3]:
from transformers import AutoTokenizer
import sentencepiece as spm

# Tải tokenizer với chế độ slow (use_fast=False để có sp_model)
tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M", use_fast=False)

# Cách 1: Lấy nội dung mô hình và lưu thành file
spm_model_proto = tokenizer.sp_model.serialized_model_proto()
with open("flores200_sacrebleu_tokenizer_spm.model", "wb") as f:
    f.write(spm_model_proto)

# Cách 2: Nếu cần đường dẫn thực tế (đối với phiên bản cũ)
try:
    print("Model path:", tokenizer.sp_model.__dict__["_model_proto_filename"])
except:
    print("Không thể lấy đường dẫn file, nhưng đã lưu thành công file .model")

Không thể lấy đường dẫn file, nhưng đã lưu thành công file .model


In [4]:
import sentencepiece as spm

# Load lại từ file đã lưu
sp = spm.SentencePieceProcessor()
sp.load("flores200_sacrebleu_tokenizer_spm.model")

# Kiểm tra
print("Tokenize ví dụ:", sp.encode_as_pieces("Xin chào thế giới"))

Tokenize ví dụ: ['▁Xin', '▁chào', '▁thế', '▁giới']


In [ ]:
import os
from unicodedata2 import *
from collections import Counter
from tqdm import tqdm
import sentencepiece as spm

try:
    import sentencepiece_model_pb2 as model
except ImportError:
    # Fallback nếu không tìm thấy module
    from sentencepiece import sentencepiece_model_pb2 as model

# Danh sách các token loại trừ (giữ nguyên)
tok_exclusion = ['<s>', '<blank>', '</s>', '<unk>', 'ace_Arab', 'ace_Latn', 'acm_Arab', 'acq_Arab', 'aeb_Arab', 'afr_Latn', 'ajp_Arab', 'aka_Latn', 'amh_Ethi', 'apc_Arab', 'arb_Arab', 'ars_Arab', 'ary_Arab', 'arz_Arab', 'asm_Beng', 'ast_Latn', 'awa_Deva', 'ayr_Latn', 'azb_Arab', 'azj_Latn', 'bak_Cyrl', 'bam_Latn', 'ban_Latn', 'bel_Cyrl', 'bem_Latn', 'ben_Beng', 'bho_Deva', 'bjn_Arab', 'bjn_Latn', 'bod_Tibt', 'bos_Latn', 'bug_Latn', 'bul_Cyrl', 'cat_Latn', 'ceb_Latn', 'ces_Latn', 'cjk_Latn', 'ckb_Arab', 'crh_Latn', 'cym_Latn', 'dan_Latn', 'deu_Latn', 'dik_Latn', 'dyu_Latn', 'dzo_Tibt', 'ell_Grek', 'eng_Latn', 'epo_Latn', 'est_Latn', 'eus_Latn', 'ewe_Latn', 'fao_Latn', 'pes_Arab', 'fij_Latn', 'fin_Latn', 'fon_Latn', 'fra_Latn', 'fur_Latn', 'fuv_Latn', 'gla_Latn', 'gle_Latn', 'glg_Latn', 'grn_Latn', 'guj_Gujr', 'hat_Latn', 'hau_Latn', 'heb_Hebr', 'hin_Deva', 'hne_Deva', 'hrv_Latn', 'hun_Latn', 'hye_Armn', 'ibo_Latn', 'ilo_Latn', 'ind_Latn', 'isl_Latn', 'ita_Latn', 'jav_Latn', 'jpn_Jpan', 'kab_Latn', 'kac_Latn', 'kam_Latn', 'kan_Knda', 'kas_Arab', 'kas_Deva', 'kat_Geor', 'knc_Arab', 'knc_Latn', 'kaz_Cyrl', 'kbp_Latn', 'kea_Latn', 'khm_Khmr', 'kik_Latn', 'kin_Latn', 'kir_Cyrl', 'kmb_Latn', 'kon_Latn', 'kor_Hang', 'kmr_Latn', 'lao_Laoo', 'lvs_Latn', 'lij_Latn', 'lim_Latn', 'lin_Latn', 'lit_Latn', 'lmo_Latn', 'ltg_Latn', 'ltz_Latn', 'lua_Latn', 'lug_Latn', 'luo_Latn', 'lus_Latn', 'mag_Deva', 'mai_Deva', 'mal_Mlym', 'mar_Deva', 'min_Latn', 'mkd_Cyrl', 'plt_Latn', 'mlt_Latn', 'mni_Beng', 'khk_Cyrl', 'mos_Latn', 'mri_Latn', 'zsm_Latn', 'mya_Mymr', 'nld_Latn', 'nno_Latn', 'nob_Latn', 'npi_Deva', 'nso_Latn', 'nus_Latn', 'nya_Latn', 'oci_Latn', 'gaz_Latn', 'ory_Orya', 'pag_Latn', 'pan_Guru', 'pap_Latn', 'pol_Latn', 'por_Latn', 'prs_Arab', 'pbt_Arab', 'quy_Latn', 'ron_Latn', 'run_Latn', 'rus_Cyrl', 'sag_Latn', 'san_Deva', 'sat_Beng', 'scn_Latn', 'shn_Mymr', 'sin_Sinh', 'slk_Latn', 'slv_Latn', 'smo_Latn', 'sna_Latn', 'snd_Arab', 'som_Latn', 'sot_Latn', 'spa_Latn', 'als_Latn', 'srd_Latn', 'srp_Cyrl', 'ssw_Latn', 'sun_Latn', 'swe_Latn', 'swh_Latn', 'szl_Latn', 'tam_Taml', 'tat_Cyrl', 'tel_Telu', 'tgk_Cyrl', 'tgl_Latn', 'tha_Thai', 'tir_Ethi', 'taq_Latn', 'taq_Tfng', 'tpi_Latn', 'tsn_Latn', 'tso_Latn', 'tuk_Latn', 'tum_Latn', 'tur_Latn', 'twi_Latn', 'tzm_Tfng', 'uig_Arab', 'ukr_Cyrl', 'umb_Latn', 'urd_Arab', 'uzn_Latn', 'vec_Latn', 'vie_Latn', 'war_Latn', 'wol_Latn', 'xho_Latn', 'ydd_Hebr', 'yor_Latn', 'yue_Hant', 'zho_Hans', 'zho_Hant', 'zul_Latn', '<pad1>', '<pad2>', '<pad3>', '<inv>']

def load_dictionary(file_path):
    """Đọc file dictionary và trả về danh sách tokens"""
    newdict2 = []
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                token = line.strip().split()[0]
                newdict2.append(token)
        return newdict2
    except Exception as e:
        print(f"Lỗi khi đọc file dictionary: {e}")
        return []

def process_model(input_model_path, output_model_path, dictionary_path):
    """Xử lý chính mô hình SentencePiece"""
    # Đọc dictionary
    newdict2 = load_dictionary(dictionary_path)
    if not newdict2:
        return False

    # Đọc mô hình SentencePiece
    try:
        with open(input_model_path, 'rb') as f:
            serializedStr = f.read()
        m = model.ModelProto()
        m.ParseFromString(serializedStr)
    except Exception as e:
        print(f"Lỗi khi đọc mô hình SentencePiece: {e}")
        return False

    # Xử lý các token
    curdict = []
    
    # Bước 1: Loại bỏ token không có trong dictionary
    for i in tqdm(range(len(m.pieces) - 1, 2, -1), desc="Đang xóa token"):
        piece = m.pieces[i]
        curdict.append(piece.piece)
        if piece.piece not in newdict2:
            hex_string = "".join("{:02x}".format(ord(c)) for c in piece.piece)
            print(f"Removing: {hex_string} từ spm model, không có trong dict. Index: {i}")
            m.pieces.pop(i)

    # Bước 2: Thêm token mới từ dictionary
    for tok in tqdm(newdict2, desc="Đang thêm token"):
        if (tok not in curdict) and (tok not in tok_exclusion):
            print(f"Adding: {tok} vào spm model")
            newtoken = m.SentencePiece()
            newtoken.piece = tok
            newtoken.score = 0  # Đặt điểm mặc định
            m.pieces.append(newtoken)

    print(f"Tổng số token sau khi xử lý: {len(m.pieces)}")

    # Lưu mô hình đã chỉnh sửa
    try:
        with open(output_model_path, 'wb') as f:
            f.write(m.SerializeToString())
        print(f"Đã lưu mô hình đã chỉnh sửa vào: {output_model_path}")
        return True
    except Exception as e:
        print(f"Lỗi khi lưu mô hình: {e}")
        return False

# Đường dẫn file
input_model = 'flores200_sacrebleu_tokenizer_spm.model'
output_model = 'flores200_sacrebleu_tokenizer_spm2.model'
dict_file = '/home/leloc/Document/USTH/Thesis/Machine_translation-Thai-Viet-/nllb-200-onmt/dictionary.txt'

# Kiểm tra file tồn tại
if not os.path.exists(input_model):
    print(f"Không tìm thấy file mô hình đầu vào: {input_model}")
elif not os.path.exists(dict_file):
    print(f"Không tìm thấy file dictionary: {dict_file}")
else:
    # Chạy quá trình xử lý
    if process_model(input_model, output_model, dict_file):
        print("Xử lý thành công!")
    else:
        print("Xử lý không thành công")

Đang xóa token:   0%|          | 190/255997 [00:00<11:47, 361.67it/s]

Removing: 85 từ spm model, không có trong dict. Index: 255860


Đang thêm token: 100%|██████████| 256214/256214 [05:52<00:00, 727.28it/s]  

Adding: ฤๅ vào spm model
Adding: ฦ vào spm model
Adding: ฦๅ vào spm model
Adding: ๏ vào spm model
Adding: ๛ vào spm model
Adding: ๚ vào spm model
Tổng số token sau khi xử lý: 256005
Đã lưu mô hình đã chỉnh sửa vào: flores200_sacrebleu_tokenizer_spm2.model
Xử lý thành công!


In [12]:
import os
import shutil
from transformers import AutoTokenizer

ORIGINAL_MODEL_NAME = "facebook/nllb-200-distilled-600M"

# 2. Đường dẫn đến file .model ĐÃ ĐƯỢC CẬP NHẬT của bạn
CUSTOM_SPM_PATH = "/home/leloc/Document/USTH/Thesis/Machine_translation-Thai-Viet-/flores200_sacrebleu_tokenizer_spm.model"

# 3. Tên thư mục đầu ra cho tokenizer mới
NEW_TOKENIZER_DIR = "./my_updated_nllb_tokenizer"
# --- KẾT THÚC CẤU HÌNH ---

def package_custom_tokenizer():
    """
    Tạo một thư mục tokenizer hoàn chỉnh từ file .model đã được cập nhật.
    """
    print(f"Bắt đầu quá trình đóng gói tokenizer mới...")

    # A. Tải tokenizer NLLB gốc để lấy các file cấu hình của nó
    print(f"1. Đang tải tokenizer gốc từ '{ORIGINAL_MODEL_NAME}' để làm mẫu...")
    try:
        original_tokenizer = AutoTokenizer.from_pretrained(ORIGINAL_MODEL_NAME)
    except Exception as e:
        print(f"Lỗi khi tải tokenizer gốc: {e}")
        return

    # B. Tạo thư mục mới và lưu cấu hình của tokenizer gốc vào đó
    if not os.path.exists(NEW_TOKENIZER_DIR):
        os.makedirs(NEW_TOKENIZER_DIR)
        print(f"2. Đã tạo thư mục mới tại: '{NEW_TOKENIZER_DIR}'")
    
    print(f"3. Đang lưu các file cấu hình gốc (tokenizer.json, config.json, etc.) vào thư mục mới...")
    original_tokenizer.save_pretrained(NEW_TOKENIZER_DIR)

    # C. GHI ĐÈ file vocabulary gốc bằng file .model đã cập nhật của bạn
    # Đây là bước quan trọng nhất!
    target_spm_file = os.path.join(NEW_TOKENIZER_DIR, "sentencepiece.bpe.model")
    print(f"4. Đang sao chép file .model tùy chỉnh của bạn vào thư mục mới...")
    print(f"   - Nguồn: {CUSTOM_SPM_PATH}")
    print(f"   - Đích: {target_spm_file}")
    
    try:
        shutil.copyfile(CUSTOM_SPM_PATH, target_spm_file)
    except Exception as e:
        print(f"Lỗi khi sao chép file .model: {e}")
        return

    print(f"\n✅ Hoàn thành! Tokenizer mới đã được đóng gói tại thư mục '{NEW_TOKENIZER_DIR}'.")
    print("Bây giờ bạn có thể sử dụng đường dẫn này trong các script khác.")

if __name__ == "__main__":
    package_custom_tokenizer()

Bắt đầu quá trình đóng gói tokenizer mới...
1. Đang tải tokenizer gốc từ 'facebook/nllb-200-distilled-600M' để làm mẫu...
3. Đang lưu các file cấu hình gốc (tokenizer.json, config.json, etc.) vào thư mục mới...
4. Đang sao chép file .model tùy chỉnh của bạn vào thư mục mới...
   - Nguồn: /home/leloc/Document/USTH/Thesis/Machine_translation-Thai-Viet-/flores200_sacrebleu_tokenizer_spm.model
   - Đích: ./my_updated_nllb_tokenizer/sentencepiece.bpe.model

✅ Hoàn thành! Tokenizer mới đã được đóng gói tại thư mục './my_updated_nllb_tokenizer'.
Bây giờ bạn có thể sử dụng đường dẫn này trong các script khác.


In [7]:
import pandas as pd
import sentencepiece as spm
from tqdm import tqdm
from collections import Counter

# 1. Load tokenizer
sp = spm.SentencePieceProcessor()
sp.load('flores200_sacrebleu_tokenizer_spm.model')

# 2. Đọc file CSV
file_path = '/home/leloc/Document/USTH/Thesis/Data/final_filtered_cleaned.csv'
df = pd.read_csv(file_path)

# 3. Hàm phát hiện UNK tokens
def detect_unk(text):
    tokens = sp.encode_as_pieces(text)
    ids = sp.encode_as_ids(text)
    unk_tokens = [token for token, id_ in zip(tokens, ids) if id_ == 3]
    return unk_tokens if unk_tokens else None

# 4. Áp dụng cho cột Thai
print("Đang kiểm tra UNK tokens...")
df['unk_tokens'] = df['Thai'].apply(detect_unk)

# 5. Lọc các dòng có UNK
unk_df = df[df['unk_tokens'].notnull()].copy()
print(f"Tìm thấy {len(unk_df)} dòng có UNK tokens")

# 6. Phân tích các ký tự UNK
all_unk = []
for tokens in unk_df['unk_tokens']:
    all_unk.extend([t.replace('▁', '') for t in tokens])

unk_counter = Counter(all_unk)

# 7. Tạo báo cáo
report = {
    'total_lines': len(df),
    'lines_with_unk': len(unk_df),
    'unique_unk_chars': len(unk_counter),
    'top_unk_chars': unk_counter.most_common(20)
}

# 8. Lưu kết quả
output_dir = '/home/leloc/Document/USTH/Thesis/Data/unk_analysis'
os.makedirs(output_dir, exist_ok=True)

# Lưu danh sách ký tự UNK
with open(f'{output_dir}/thai_unk_chars.txt', 'w', encoding='utf-8') as f:
    f.write("=== DANH SÁCH KÝ TỰ UNK ===\n")
    for char, count in unk_counter.most_common():
        f.write(f"'{char}': {count} lần\n")

# Lưu các dòng có UNK
unk_df.to_csv(f'{output_dir}/lines_with_unk.csv', index=False)

# Lưu thống kê tổng quan
with open(f'{output_dir}/summary.txt', 'w', encoding='utf-8') as f:
    f.write(f"Tổng số dòng: {report['total_lines']}\n")
    f.write(f"Số dòng có UNK: {report['lines_with_unk']}\n")
    f.write(f"Số ký tự UNK duy nhất: {report['unique_unk_chars']}\n\n")
    f.write("Top 20 ký tự UNK:\n")
    for char, count in report['top_unk_chars']:
        f.write(f"'{char}': {count} lần\n")

print(f"Đã lưu kết quả vào thư mục: {output_dir}")

Đang kiểm tra UNK tokens...
Tìm thấy 0 dòng có UNK tokens
Đã lưu kết quả vào thư mục: /home/leloc/Document/USTH/Thesis/Data/unk_analysis


In [4]:
import pandas as pd
import re

# Đường dẫn file
input_file = '/home/leloc/Document/USTH/Thesis/Data/final_filtered_v2.csv'
output_file = '/home/leloc/Document/USTH/Thesis/Data/final_filtered_cleaned.csv'

# Phạm vi Unicode tiếng Thái (bao gồm cả dấu câu, số trong tiếng Thái)
THAI_CHAR_RANGE = r'[\u0E00-\u0E7F\s\d\u0E3F\u0E4F\u0E5A\u0E5B.,!?;:"\'\-\(\)\[\]\{\}]'

def is_valid_thai(text):
    """Kiểm tra nếu text chỉ chứa ký tự tiếng Thái hợp lệ"""
    if not isinstance(text, str):
        return False
    # Xóa khoảng trắng và dấu câu chung
    cleaned = re.sub(r'[\s\d.,!?;:"\'\-\(\)\[\]\{\}]', '', text)
    # Kiểm tra từng ký tự còn lại
    return all('\u0E00' <= char <= '\u0E7F' for char in cleaned)

# Đọc dữ liệu
print("📖 Đang đọc file CSV...")
df = pd.read_csv(input_file)
original_count = len(df)
print(f"Tổng số dòng ban đầu: {original_count}")

# Lọc dữ liệu
print("🔍 Đang lọc dữ liệu...")
valid_thai_mask = df['Thai'].apply(is_valid_thai)
cleaned_df = df[valid_thai_mask].copy()

# Thống kê
removed_count = original_count - len(cleaned_df)
print(f"✅ Số dòng hợp lệ: {len(cleaned_df)}")
print(f"❌ Số dòng bị loại bỏ: {removed_count}")

# Lưu file mới
cleaned_df.to_csv(output_file, index=False)
print(f"💾 Đã lưu file đã làm sạch vào: {output_file}")

# Lưu các dòng bị loại để kiểm tra (tùy chọn)
if removed_count > 0:
    removed_df = df[~valid_thai_mask]
    removed_file = '/home/leloc/Document/USTH/Thesis/Data/removed_non_thai_lines.csv'
    removed_df.to_csv(removed_file, index=False)
    print(f"📝 Đã lưu danh sách dòng bị loại vào: {removed_file}")

📖 Đang đọc file CSV...
Tổng số dòng ban đầu: 298996
🔍 Đang lọc dữ liệu...
✅ Số dòng hợp lệ: 281764
❌ Số dòng bị loại bỏ: 17232
💾 Đã lưu file đã làm sạch vào: /home/leloc/Document/USTH/Thesis/Data/final_filtered_cleaned.csv
📝 Đã lưu danh sách dòng bị loại vào: /home/leloc/Document/USTH/Thesis/Data/removed_non_thai_lines.csv
